# Multirotor: Drive → 추출/무차원화 → 학습 → 예측/평가
런타임에서 GPU를 선택한 뒤 위에서 아래로 실행하세요. 실제 Drive 경로는 설정 셀에서 바꿉니다.
추출 단계에서 무차원화도 수행하므로 normalization.py를 따로 실행하지 않습니다.
원본은 OpenFOAM 케이스 구조(`constant/polyMesh`, 시간 폴더의 `U`)를 유지해야 합니다.
CSV `folder`는 원본 루트 기준 상대경로입니다. 예: `cases/case01` 또는 `001`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import shutil
import subprocess
import sys
import uuid

# 이 세 경로를 실제 Drive 위치에 맞추세요.
DRIVE_DATA = Path('/content/drive/MyDrive/multirotor_data')
DRIVE_CSV = DRIVE_DATA / 'cases.csv'  # case.csv라면 파일명 변경
DRIVE_RESULTS = Path('/content/drive/MyDrive/multirotor_results')
EPOCHS = 100
BATCH_SIZE = 4

REPO = Path('/content/multirotor')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/multirotor_runs') / RUN_ID
RUN_RESULTS = DRIVE_RESULTS / RUN_ID
RAW = WORK / 'raw'
CSV = WORK / 'cases.csv'
DATASET = WORK / 'dataset_nd'
MODELS = RUN_RESULTS / 'models'
RESULTS = RUN_RESULTS / 'results'

def run(*args):
    subprocess.run([str(x) for x in args], check=True,
                   env={**os.environ, 'MPLBACKEND': 'Agg'})

if not REPO.exists():
    run('git', 'clone', 'https://github.com/jo-ban/multirotor.git', REPO)
else:
    run('git', '-C', REPO, 'pull', '--ff-only')
CODE = REPO / 'work_multirotor'
run(sys.executable, '-m', 'pip', 'install', '-r', CODE / 'requirements.txt')
print('이번 실행 결과:', RUN_RESULTS)


원본을 Colab 작업 공간으로 복사합니다. 데이터 크기만큼 로컬 디스크 여유가 필요합니다. 결과 폴더는 원본 폴더 바깥으로 설정하세요. 매 실행마다 새 폴더를 사용합니다.

In [ ]:
if not DRIVE_DATA.is_dir() or not DRIVE_CSV.is_file():
    raise FileNotFoundError('DRIVE_DATA와 DRIVE_CSV를 확인하세요.')
if DRIVE_RESULTS.resolve().is_relative_to(DRIVE_DATA.resolve()):
    raise ValueError('DRIVE_RESULTS는 DRIVE_DATA 바깥에 설정하세요.')
WORK.mkdir(parents=True)
shutil.copytree(DRIVE_DATA, RAW)
shutil.copy2(DRIVE_CSV, CSV)
RUN_RESULTS.mkdir(parents=True)
shutil.copy2(CSV, RUN_RESULTS / 'cases.csv')
revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(RUN_RESULTS / 'code_revision.txt').write_text(revision)


## 1. 추출 및 무차원화
케이스 하나라도 실패하면 셀이 실패합니다. 오류를 해결하고 추출을 완료한 다음 학습하세요.

In [ ]:
run(sys.executable, CODE / 'extract_nd.py',
    '--csv', CSV, '--data-root', RAW, '--output-root', DATASET)
# 추출 결과도 Drive에 보관합니다.
shutil.copytree(DATASET, RUN_RESULTS / 'dataset_nd', dirs_exist_ok=True)


## 2. 학습
데이터셋은 Colab 로컬에서 읽습니다. U/V/W 모델은 각 성분 학습이 끝날 때 Drive에 저장됩니다. 학습 도중 세션이 종료되면 현재 성분의 학습은 다시 해야 합니다.

In [ ]:
run(sys.executable, CODE / 'train_nd.py',
    '--csv', CSV, '--data-root', DATASET, '--model-dir', MODELS,
    '--epochs', EPOCHS, '--batch-size', BATCH_SIZE)


## 3. 예측
아래 물리 조건을 원하는 값으로 바꾸세요. 길이는 m, 디스크 로딩은 N/m²입니다. 결과 NPZ와 PNG가 Drive에 저장됩니다.

In [ ]:
run(sys.executable, CODE / 'predict_nd.py',
    '--model-dir', MODELS, '--output', RESULTS / 'prediction.npz',
    '--rotor-spacing', 1.25, '--rotor-diameter', 1.0,
    '--disk-loading', 153.22, '--rotor-z', 2.0, '--ground-z', 0.0,
    '--no-show')
from IPython.display import display, Image
display(Image(filename=str(RESULTS / 'prediction.png')))


## 4. 검증 케이스 오차 평가 (선택)
CSV의 `type=V` 케이스만 평가하며 해당 케이스는 학습에서 제외됩니다.

In [ ]:
import pandas as pd
table = pd.read_csv(CSV)
if 'type' in table and table['type'].astype(str).str.upper().eq('V').any():
    run(sys.executable, CODE / 'evaluate_error.py',
        '--csv', CSV, '--data-root', DATASET, '--model-dir', MODELS,
        '--output-dir', RESULTS / 'evaluation', '--no-show')
    display(pd.read_csv(RESULTS / 'evaluation' / 'validation_errors_cyl2rd.csv'))
else:
    print('type=V 케이스가 없어 평가를 건너뜁니다.')
print('Drive 저장 위치:', RUN_RESULTS)
